# Etapa 0 — Preparação e Tratamento dos Dados

**Base de dados:** `ElectricityLoadDiagrams20112014` (UCI Machine Learning Repository, ID 321, licença CC BY 4.0).  
**Objetivo:** Auditar o arquivo bruto (`Dados/bruto/LD2011_2014.txt`), tratar zeros estruturais (pré-adesão e pós-desligamento), corrigir anomalias de horário de verão (DST de março e outubro), interpolar falhas curtas, regularizar o índice temporal (`15min`, `1h` e `1D`), selecionar clientes completos e gerar os arquivos tratados em `Dados/tratado/` e figuras diagnósticas em `Documentacao/figuras/`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

# Resolve a raiz do projeto independentemente de o notebook ser executado de Codigo/ ou da raiz
RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == "Codigo" else Path.cwd()
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from Codigo.src.preparacao import (
    SEMENTE_PADRAO,
    agregar_e_filtrar_series,
    carregar_dados_brutos,
    diagnosticar_dados_brutos,
    gerar_figuras_preparacao,
    salvar_saidas_preparacao,
    tratar_series_15min,
)

np.random.seed(SEMENTE_PADRAO)
print(f"Raiz do projeto: {RAIZ_PROJETO}")

## 1. Carregamento e Auditoria dos Dados Brutos

Os timestamps originais representam o fim de cada intervalo de 15 minutos (`2011-01-01 00:15:00` a `2015-01-01 00:00:00`). Ao subtrair 15 minutos, alinhamos o índice ao início do bloco (`2011-01-01 00:00:00` a `2014-12-31 23:45:00`).

In [ ]:
df_bruto = carregar_dados_brutos(RAIZ_PROJETO / "Dados/bruto/LD2011_2014.txt")
diag_bruto = diagnosticar_dados_brutos(df_bruto)

print(f"Dimensões: {df_bruto.shape} | Frequência: {df_bruto.index.freqstr}")
display(df_bruto.iloc[:5, :6])
print(json.dumps({k: v for k, v in diag_bruto.items() if not isinstance(v, dict)}, indent=2, ensure_ascii=False))

## 2. Limpeza, Correção de DST e Interpolação Temporal

1. Zeros pré-adesão e pós-desligamento são convertidos em `NaN` (ausência de medição).
2. No DST de março (2012–2014), toda a janela da hora suprimida em clientes ativos é convertida em `NaN` (removendo zeros artificiais e valores residuais espúrios).
3. No DST de outubro (2012–2014), toda a janela da hora repetida em clientes ativos é dividida por `2.0` (2011 manteve registro regular e é preservado).
4. Lacunas curtas internas (`<= 8` passos de 15 min, i.e. `<= 2h`) são interpoladas com `method='time'`.

In [ ]:
df_tratado_15min, df_resumo_clientes, resumo_tratamento = tratar_series_15min(df_bruto)
print(json.dumps(resumo_tratamento, indent=2, ensure_ascii=False))
display(df_resumo_clientes.head(10))

## 3. Regularização Temporal (`1h` e `1D`) e Seleção de Clientes Completos

Reamostramos para `1h` (`mean`, preservando kW = kWh/h) e `1D` (`mean`), e filtramos os clientes ativos em toda a janela `2011-01-01` a `2014-12-31` com taxa de zeros internos longos `<= 1%`.

In [ ]:
agregado = agregar_e_filtrar_series(df_tratado_15min, df_resumo_clientes)
print(json.dumps(agregado["resumo_agregacao"], indent=2, ensure_ascii=False))
display(agregado["df_horario_completos"].iloc[:5, :6])

## 4. Figuras Diagnósticas (Antes/Depois) e Persistência

In [ ]:
figuras = gerar_figuras_preparacao(
    df_bruto=df_bruto,
    df_tratado_15min=df_tratado_15min,
    df_horario_completos=agregado["df_horario_completos"],
    df_diario_completos=agregado["df_diario_completos"],
    df_resumo_clientes=df_resumo_clientes,
    diretorio_figuras=RAIZ_PROJETO / "Documentacao/figuras",
)

metricas = {
    "semente": SEMENTE_PADRAO,
    "diagnostico_bruto": diag_bruto,
    "tratamento_15min": resumo_tratamento,
    "agregacao_e_selecao": agregado["resumo_agregacao"],
    "figuras_geradas": [str(Path(f).relative_to(RAIZ_PROJETO)) for f in figuras],
}

saidas = salvar_saidas_preparacao(
    agregado=agregado,
    df_resumo_clientes=df_resumo_clientes,
    metricas=metricas,
    dir_tratado=RAIZ_PROJETO / "Dados/tratado",
    dir_resultados=RAIZ_PROJETO / "Resultados",
)

for fig_path in figuras:
    display(Image(filename=fig_path))